# Seoul Bike - Random Forest Optimization

Prediction of hourly bicycle rentals using weather, time, and calendar features.


## 0. Data preparation and chronological split

- **Train (70%)** - used to fit the model.
- **Validation (15%)** - used to choose the best settings and features for the model.
- **Test (15%)** - used to evaluate the selected model on a later time period.

Observations are sorted by date and hour before splitting. This keeps later observations out of the earlier training period.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# Load and prepare data
df = pd.read_csv("../data/SeoulBikeData.csv", encoding="latin1")
df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y")
df = (
    df[df["Functioning Day"] == "Yes"]
    .sort_values(["Date", "Hour"])
    .reset_index(drop=True)
)

df["DayOfWeek"] = df["Date"].dt.day_of_week
df["IsHoliday"] = (df["Holiday"] == "Holiday").astype(int)

# Original selected features
base_features = [
    "Hour",
    "Temperature(°C)",
    "Rainfall(mm)",
    "Humidity(%)",
    "IsHoliday",
    "DayOfWeek",
]

# Weather features suggested during the group meeting
weather_features = [
    "Temperature(°C)",
    "Humidity(%)",
    "Wind speed (m/s)",
    "Visibility (10m)",
    "Dew point temperature(°C)",
    "Solar Radiation (MJ/m2)",
    "Rainfall(mm)",
    "Snowfall (cm)",
]

feature_sets = {
    "Selected 6 features": base_features,
    "Weather + Hour (9)": ["Hour"] + weather_features,
    "Weather + Hour + calendar (11)": ["Hour"] + weather_features + ["DayOfWeek", "IsHoliday"],
}

x = df[base_features]
y = df["Rented Bike Count"]

# Keep later observations out of training
train_end = int(len(df) * 0.70)
val_end = int(len(df) * 0.85)

x_train = x.iloc[:train_end]
x_val = x.iloc[train_end:val_end]
y_train = y.iloc[:train_end]
y_val = y.iloc[train_end:val_end]
y_test = y.iloc[val_end:]

print(f"Train: {train_end}, Validation: {val_end - train_end}, Test: {len(df) - val_end}")

In [ ]:
# Visualize the chronological split of the original 8,465 functioning-hour observations
fig, ax = plt.subplots(figsize=(12, 2.4))
segments = [
    (0, train_end, 'Train (70%)', '#4f6b9a'),
    (train_end, val_end - train_end, 'Validation (15%)', '#d69f45'),
    (val_end, len(df) - val_end, 'Test (15%)', '#69a587'),
]
for start, length, label, color in segments:
    ax.barh(0, length, left=start, height=0.55, label=label, color=color)

ax.set_yticks([])
ax.set_xlim(0, len(df))
ax.set_xlabel('Observations ordered chronologically')
ax.set_title('Chronological Train / Validation / Test split')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.42), ncol=3, frameon=False)
plt.tight_layout()
plt.show()

for name, start, end in [
    ('Train', 0, train_end),
    ('Validation', train_end, val_end),
    ('Test', val_end, len(df)),
]:
    print(f"{name}: {df.iloc[start]['Date']:%Y-%m-%d} to {df.iloc[end-1]['Date']:%Y-%m-%d}")


**Random vs. chronological split:** A separate experiment using a randomly shuffled 80/20 Train/Test split obtained **MAE = 151.79**. A random split places observations from different months in both sets, while a chronological split tests the model on later periods. These results are **not a controlled comparison of split methods**, because the models also used different hyperparameters.


## 1. Initial Random Forest — Overfitting

The initial model uses 100 trees with unrestricted depth. Its performance is evaluated separately on Train and Validation.

- **MAE** — average absolute prediction error, in bicycles per hour; lower is better.
- **RMSE** — prediction error that penalizes large misses more strongly; lower is better.
- **R²** — goodness of fit relative to predicting the mean; higher is better.


In [ ]:
def regression_metrics(actual, predicted):
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": root_mean_squared_error(actual, predicted),
        "R²": r2_score(actual, predicted),
    }

forest = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
forest.fit(x_train, y_train)

initial_results = pd.DataFrame([
    {"Split": "Train", **regression_metrics(y_train, forest.predict(x_train))},
    {"Split": "Validation", **regression_metrics(y_val, forest.predict(x_val))},
])
display(initial_results.round(3))

**Observation:** Train MAE is approximately **39.41**, compared with **183.28** on Validation. This gap suggests possible **overfitting**: the model fits training observations much better than unseen observations. Differences between time periods may also contribute.


## 2. Optimizing tree depth (`max_depth`)

Tree depth controls model complexity. Increasing depth can reduce Train MAE without improving Validation MAE. The optimal depth is therefore selected using the Validation results.


In [ ]:
depth_results = []

for depth in range(4, 21):
    model = RandomForestRegressor(
        n_estimators=100,
        max_depth=depth,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(x_train, y_train)
    depth_results.append({
        "Depth": depth,
        "Train MAE": mean_absolute_error(y_train, model.predict(x_train)),
        "Validation MAE": mean_absolute_error(y_val, model.predict(x_val)),
    })

depth_results = pd.DataFrame(depth_results)
best_depth = int(depth_results.loc[depth_results["Validation MAE"].idxmin(), "Depth"])

plt.figure(figsize=(15, 4))
plt.plot(depth_results["Depth"], depth_results["Train MAE"], marker="o", label="Train")
plt.plot(depth_results["Depth"], depth_results["Validation MAE"], marker="o", label="Validation")
plt.title("Random Forest — MAE vs Max Depth")
plt.xlabel("Max depth")
plt.ylabel("MAE (bikes/hour)")
plt.xticks(depth_results["Depth"])
plt.legend()
plt.grid(True)
plt.show()

print(f"Best depth: {best_depth}")
print(f"Best Validation MAE: {depth_results['Validation MAE'].min():.2f}")

**Observation:** The lowest Validation MAE occurs at **max_depth = 14**. Beyond this depth, Train MAE continues to decrease while Validation MAE changes very little.


## 3. Optimizing the number of trees

Increasing the number of trees requires more computation. Validation MAE improves only slightly beyond approximately **50 trees**.


In [ ]:
count_results = []

for count in [10, 20, 50, 100, 150, 200, 300]:
    model = RandomForestRegressor(
        n_estimators=count,
        max_depth=best_depth,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(x_train, y_train)
    count_results.append({
        "Trees": count,
        "Train MAE": mean_absolute_error(y_train, model.predict(x_train)),
        "Validation MAE": mean_absolute_error(y_val, model.predict(x_val)),
    })

count_results = pd.DataFrame(count_results)

plt.figure(figsize=(15, 4))
plt.plot(count_results["Trees"], count_results["Train MAE"], marker="o", label="Train")
plt.plot(count_results["Trees"], count_results["Validation MAE"], marker="o", label="Validation")
plt.title("Random Forest — MAE vs Number of Trees")
plt.xlabel("Number of trees")
plt.ylabel("MAE (bikes/hour)")
plt.xticks(count_results["Trees"])
plt.legend()
plt.grid(True)
plt.show()

chosen_count = 50
print(f"Selected number of trees: {chosen_count}")
display(count_results.round(2))

**Decision:** **50 trees** provide a practical balance between prediction accuracy and computational cost, despite slightly lower Validation MAE for some larger forests.


## 4. Comparing feature sets

Three feature sets are evaluated with the **same chronological Train/Validation split** and **the same Random Forest hyperparameters**:

- **6 selected features:** Hour, Temperature, Rainfall, Humidity, IsHoliday, DayOfWeek.
- **9 weather + time features:** eight weather measurements and Hour.
- **11 weather + time + calendar features:** the previous nine, plus DayOfWeek and IsHoliday.

The larger sets incorporate weather measurements proposed by other group members.


In [ ]:
feature_results = []

for name, columns in feature_sets.items():
    x_current = df[columns]
    model = RandomForestRegressor(
        n_estimators=chosen_count,
        max_depth=best_depth,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(x_current.iloc[:train_end], y_train)
    validation_pred = model.predict(x_current.iloc[train_end:val_end])
    feature_results.append({
        "Feature set": name,
        "Features": len(columns),
        **regression_metrics(y_val, validation_pred),
    })

feature_comparison = pd.DataFrame(feature_results).sort_values("MAE").reset_index(drop=True)
display(feature_comparison.round(3))

**Observations:**

- **MAE:** Almost unchanged between 6 and 11 features (**181.25 vs. 180.63**). The additional five features barely improve average prediction accuracy.
- **RMSE:** Improves from **286.44 to 271.26**. The 11-feature model has a smaller overall squared-error penalty, suggesting fewer or less severe large errors.
- **R²:** Increases from **0.811 to 0.831**, indicating a better fit to changes in rental demand on Validation.

The **11-feature set** has the lowest Validation MAE and is selected for the final evaluation.


## 5. Final evaluation on Test

The selected model is refitted on **Train + Validation (85%)**, then evaluated on the later **Test period (15%)**. The Test results are not used for further parameter selection.


In [ ]:
best_feature_set = feature_comparison.loc[0, "Feature set"]
final_features = feature_sets[best_feature_set]
x_final = df[final_features]

final_forest = RandomForestRegressor(
    n_estimators=chosen_count,
    max_depth=best_depth,
    random_state=42,
    n_jobs=-1,
)
final_forest.fit(x_final.iloc[:val_end], y.iloc[:val_end])
final_predictions = final_forest.predict(x_final.iloc[val_end:])

final_result = pd.DataFrame([{
    "Selected features": best_feature_set,
    "Max depth": best_depth,
    "Trees": chosen_count,
    **regression_metrics(y_test, final_predictions),
}])
display(final_result.round(3))

**Observation:** Test results (**MAE = 261.02**, **RMSE = 352.94**, **R² = 0.592**) are worse than the Validation results (**180.63**, **271.26**, **0.831**). Seasonal differences or other changes between periods are possible explanations, but have not been established. Good Validation performance does not guarantee equally good performance on later data.


## 6. Feature importance

Built-in Random Forest feature importance summarizes how much each input contributes to the forest's tree splits.


In [ ]:
importance = pd.Series(
    final_forest.feature_importances_,
    index=final_features
).sort_values()

plt.figure(figsize=(10, 6))
importance.plot.barh()

plt.title("Random Forest — Feature Importance")
plt.xlabel("Feature Importance")
plt.ylabel("")
plt.tight_layout()
plt.show()

**Observation:** Temperature and Hour account for about **70%** of the built-in feature importance. Solar Radiation and Humidity also contribute.  
Feature importance shows how much the model uses each feature, but not exactly how much the model depends on it.


## 7. Actual vs. Predicted

Actual hourly bicycle rentals are compared with predictions from the final Random Forest for the first **168 functioning-hour observations** in Test.


In [ ]:
# Compare the final model with reality for the first 168 test observations
n = min(168, len(y_test))

plt.figure(figsize=(14, 5))
plt.plot(y_test.iloc[:n].to_numpy(), label='Actual', linewidth=2)
plt.plot(final_predictions[:n], label='Predicted', linewidth=2, alpha=0.8)
plt.title('Random Forest — Actual vs Predicted (Test)')
plt.xlabel('Sequential test observations (functioning hours)')
plt.ylabel('Rented Bike Count')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


**Observations:** The model captures the general daily demand pattern, but its predictions are smoother than the actual values. High rental peaks are often underestimated, while low-demand periods are sometimes overestimated. This helps explain the relatively high Test MAE and RMSE.
